# Textual Inversion su SDXL — Test

> **Prerequisito**: esegui prima il training con `experiments/ti/train/launcher.py`
> e assicurati che `models/trained/ti_cat_v1/learned_embeds_final.safetensors` esista.

In [1]:
import torch
from diffusers import StableDiffusionXLPipeline
from safetensors.torch import load_file
import os
import yaml

# Carica configurazione personale (separata dal config condiviso)
with open("/homes/ktotaro/cvcs2026/config_ti.yaml", "r") as f:
    config = yaml.safe_load(f)

In [2]:
MODEL_PATH        = config["paths"]["base_model_dir"]
TI_MODEL_DIR      = config["paths"]["tiv1_model_dir"]
TI_WEIGHTS_FILE   = config["weights_file"]["ti_weights"]
OUTPUT_DIR        = config["paths"]["ti_output_test_dir"]
PLACEHOLDER_TOKEN = config["hyperparameters"]["placeholder_token"]

EMBEDDING_PATH = os.path.join(TI_MODEL_DIR, TI_WEIGHTS_FILE)

# Stesso set di prompt usato in test_lora.ipynb
prompts = [
    f"A photo of {PLACEHOLDER_TOKEN} on the moon",                  # Personalizzazione (Soggetto + Contesto)
    f"A Renaissance oil painting of {PLACEHOLDER_TOKEN}",           # Personalizzazione (Soggetto + Stile)
    f"A photo of {PLACEHOLDER_TOKEN} on a street",
    f"A photo of {PLACEHOLDER_TOKEN} with a man",
    f"App icon of {PLACEHOLDER_TOKEN}",
    f"A {PLACEHOLDER_TOKEN} themed lunchbox",
    "A photo of a golden retriever dog",                            # Preservazione (Il modello ricorda altri animali?)
    "A beautiful mountain landscape at sunset",                     # Preservazione (C'è 'leakage' del gatto?)
    "A photo of a cat",                                             # Prior Preservation (Sa fare ancora gatti generici?)
    "A photo of a beautiful cat"
]

### Load model

In [3]:
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

# Aggiunge token a entrambi i tokenizer e ridimensiona le embedding table
pipe.tokenizer.add_tokens([PLACEHOLDER_TOKEN])
pipe.tokenizer_2.add_tokens([PLACEHOLDER_TOKEN])
pipe.text_encoder.resize_token_embeddings(len(pipe.tokenizer))
pipe.text_encoder_2.resize_token_embeddings(len(pipe.tokenizer_2))

# Carica e inietta gli embedding appresi
token_id_one = pipe.tokenizer.convert_tokens_to_ids(PLACEHOLDER_TOKEN)
token_id_two = pipe.tokenizer_2.convert_tokens_to_ids(PLACEHOLDER_TOKEN)
tensors = load_file(EMBEDDING_PATH)
with torch.no_grad():
    pipe.text_encoder.get_input_embeddings().weight[token_id_one] = (
        tensors["clip_l"].to(dtype=pipe.text_encoder.dtype, device="cuda")
    )
    pipe.text_encoder_2.get_input_embeddings().weight[token_id_two] = (
        tensors["clip_g"].to(dtype=pipe.text_encoder_2.dtype, device="cuda")
    )

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


### Generate images

In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

for i, prompt in enumerate(prompts):
    print(f"Generating image for prompt {i+1}/{len(prompts)}: '{prompt}'")
    image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
    filename = f"{OUTPUT_DIR}/prompt_{i+1}.png"
    image.save(filename)

Generating image for prompt 1/10: 'A photo of <cat2> on the moon'


  0%|          | 0/30 [00:00<?, ?it/s]

/homes/ktotaro/cvcs2026/venv/lib/python3.11/site-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


Generating image for prompt 2/10: 'A Renaissance oil painting of <cat2>'


  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

Generating image for prompt 9/10: 'A photo of a cat'


  0%|          | 0/30 [00:00<?, ?it/s]

Generating image for prompt 10/10: 'A photo of a beautiful cat'


  0%|          | 0/30 [00:00<?, ?it/s]